In [1]:
# If needed, uncomment in Colab/your Jupyter environment:
# !pip install -U transformers datasets accelerate evaluate seqeval scikit-learn

import os
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np
import torch
from datasets import Dataset as HFDataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification, TrainingArguments, Trainer)
import evaluate
from sklearn.metrics import confusion_matrix

!pip install -U transformers


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Config:
    
    dataset_dir = "/home/jovyan/work/EntityExtractionDataSets/"  # folder with train.txt, dev.txt, test.txt
    train_file = os.path.join(dataset_dir, "train.txt")
    dev_file   = os.path.join(dataset_dir, "dev.txt")
    test_file  = os.path.join(dataset_dir, "test.txt")

    model_name = "bert-base-cased"
    max_length = 128
    freeze_encoder = False

    seed = 42
    per_device_train_batch_size = 16
    per_device_eval_batch_size = 32
    learning_rate = 5e-5
    num_train_epochs = 5
    weight_decay = 0.01
    gradient_accumulation_steps = 1
    fp16 = torch.cuda.is_available()

    output_dir = "./outputs_ner"
    save_total_limit = 2
    evaluation_strategy = "epoch"
    save_strategy = "epoch"
    load_best_model_at_end = True
    metric_for_best_model = "f1"
    greater_is_better = True

cfg = Config()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

os.makedirs(cfg.output_dir, exist_ok=True)
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

Using device: cuda


In [3]:
# Robust CoNLL/CoNLL-like loader:
# - Accepts tabs OR any whitespace between token and label
# - Uses the LAST column as the label (so "token   LABEL" or "token\tLABEL" both work)
# - Blank lines (or a line "[SENT_BREAK]") mark sentence boundaries
# - Skips lines that don't have at least 2 columns, but logs a warning once

from typing import List, Dict

def read_conll(path: str) -> Dict[str, List[List[str]]]:
    tokens: List[List[str]] = []
    labels: List[List[str]] = []
    cur_tokens: List[str] = []
    cur_labels: List[str] = []

    warned = False  # only warn once for malformed lines
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            # Sentence boundary
            if not line or line == "[SENT_BREAK]":
                if cur_tokens:
                    tokens.append(cur_tokens)
                    labels.append(cur_labels)
                    cur_tokens, cur_labels = [], []
                continue

            # Split on ANY whitespace; accept 2+ columns and take LAST as label
            parts = line.split()
            if len(parts) >= 2:
                tok = parts[0]            # token is the first column
                lab = parts[-1]           # label is the last column (handles extra columns safely)
                cur_tokens.append(tok)
                cur_labels.append(lab)
            else:
                if not warned:
                    print(f"[read_conll] Skipping malformed line (expected 'TOKEN LABEL'): {raw!r}")
                    warned = True
                continue

    # Flush final sentence if file doesn’t end with a blank line
    if cur_tokens:
        tokens.append(cur_tokens)
        labels.append(cur_labels)

    return {"tokens": tokens, "labels": labels}

# Re-run loading using the robust parser
train_data = read_conll(cfg.train_file)
dev_data   = read_conll(cfg.dev_file)
test_data  = read_conll(cfg.test_file)

# Build label set from train split
all_labels = sorted({lab for seq in train_data["labels"] for lab in seq})
label2id = {lab: i for i, lab in enumerate(all_labels)}
id2label = {i: lab for lab, i in label2id.items()}

print(f"Sentences — train/dev/test: {len(train_data['tokens'])}/{len(dev_data['tokens'])}/{len(test_data['tokens'])}")
print(f"Labels: {all_labels}")


Sentences — train/dev/test: 1144/308/348
Labels: ['ATK', 'O']


In [4]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def tokenize_and_align_labels(examples: Dict[str, List[List[str]]]) -> Dict[str, Any]:
    tokenized = tokenizer(examples["tokens"], is_split_into_words=True, truncation=True, max_length=cfg.max_length)
    aligned_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[labels[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized

In [5]:
# Peek at the first parsed sentence from dev split
print(dev_data["tokens"][0][:30])
print(dev_data["labels"][0][:30])

# Count any weird labels that snuck in
from collections import Counter
print("Label counts (train):", Counter(l for seq in train_data["labels"] for l in seq).most_common()[:10])


['Since', 'then', ',', 'the', 'implant', '’', 's', 'functionality', 'has', 'been', 'improving', 'and', 'remarkable', 'new', 'features', 'implemented', ',', 'such', 'as', 'the', 'ability', 'to', 'record', 'audio', 'surroundings', 'via', 'the', 'microphone', 'when', 'an']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ATK', 'ATK', 'ATK', 'ATK', 'ATK', 'ATK', 'ATK', 'ATK']
Label counts (train): [('ATK', 13018), ('O', 12425)]


In [6]:
def to_hf_dataset(data_dict: Dict[str, List[List[str]]]) -> HFDataset:
    return HFDataset.from_dict({
        "tokens": data_dict["tokens"],
        "labels": data_dict["labels"],
    })

raw_train = to_hf_dataset(train_data)
raw_dev   = to_hf_dataset(dev_data)
raw_test  = to_hf_dataset(test_data)

tokenized_train = raw_train.map(tokenize_and_align_labels, batched=True)
tokenized_dev   = raw_dev.map(tokenize_and_align_labels, batched=True)
tokenized_test  = raw_test.map(tokenize_and_align_labels, batched=True)

ds = DatasetDict(train=tokenized_train, validation=tokenized_dev, test=tokenized_test)
ds

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 348/348 [00:00<00:00, 15397.62 examples/s]


DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1144
    })
    validation: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 308
    })
    test: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 348
    })
})

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    cfg.model_name,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
)

if cfg.freeze_encoder:
    for name, param in model.named_parameters():
        if not any(h in name for h in ["classifier", "lm_head"]):
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Frozen encoder. Trainable params: {trainable}/{total}")
else:
    print("Fine-tuning full model.")

Loading weights: 100%|████████████████████████| 197/197 [00:00<00:00, 592.98it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can b

Fine-tuning full model.


In [8]:
# Step 7 - METRICS + DATA COLLATOR

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=-1)

    true_labels, true_predictions = [], []
    for pred, lab in zip(predictions, labels):
        cur_preds, cur_labels = [], []
        for p_i, l_i in zip(pred, lab):
            if l_i == -100:
                continue
            cur_preds.append(id2label[p_i])
            cur_labels.append(id2label[l_i])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results.get("overall_precision", 0.0),
        "recall": results.get("overall_recall", 0.0),
        "f1": results.get("overall_f1", 0.0),
        "accuracy": results.get("overall_accuracy", 0.0),
    }

data_collator = DataCollatorForTokenClassification(tokenizer)


In [9]:
# Trainer Setup

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    weight_decay=cfg.weight_decay,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    eval_strategy=cfg.evaluation_strategy,
    save_strategy=cfg.save_strategy,
    load_best_model_at_end=cfg.load_best_model_at_end,
    metric_for_best_model=cfg.metric_for_best_model,
    greater_is_better=cfg.greater_is_better,
    logging_dir=os.path.join(cfg.output_dir, "logs"),
    save_total_limit=cfg.save_total_limit,
    fp16=cfg.fp16,
    report_to=["none"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [12]:
# cell 9 - TRAIN

train_result = trainer.train()
trainer.save_model(cfg.output_dir)
metrics = trainer.evaluate(ds["validation"])
metrics


/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.300498,0.481203,0.602353,0.535005,0.873680
2,No log,0.313151,0.521825,0.618824,0.566200,0.872976
3,No log,0.347124,0.537849,0.635294,0.582524,0.881144
4,No log,0.413048,0.516765,0.616471,0.562232,0.881003
5,No log,0.491316,0.521298,0.604706,0.559913,0.874525


/opt/conda/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ATK seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/conda/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ATK seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/conda/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ATK seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/co

/opt/conda/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ATK seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


{'eval_loss': 0.34712398052215576,
 'eval_precision': 0.5378486055776892,
 'eval_recall': 0.6352941176470588,
 'eval_f1': 0.5825242718446602,
 'eval_accuracy': 0.8811435009153641,
 'eval_runtime': 5.2019,
 'eval_samples_per_second': 59.209,
 'eval_steps_per_second': 1.922,
 'epoch': 5.0}

In [13]:
# CELL 10 EVALUATION + CONFUSION MATRIX

test_metrics = trainer.evaluate(ds["test"])
print("Test metrics:", test_metrics)

predictions, labels, _ = trainer.predict(ds["test"])
pred_ids = np.argmax(predictions, axis=-1)
gold_ids, pred_ids_flat = [], []
for p_row, l_row in zip(pred_ids, labels):
    for p_i, l_i in zip(p_row, l_row):
        if l_i == -100:
            continue
        gold_ids.append(l_i)
        pred_ids_flat.append(p_i)

labels_order = list(range(len(all_labels)))
cm = confusion_matrix(gold_ids, pred_ids_flat, labels=labels_order)
np.save(os.path.join(cfg.output_dir, "confusion_matrix.npy"), cm)
print("Saved confusion matrix to:", os.path.join(cfg.output_dir, "confusion_matrix.npy"))


Test metrics: {'eval_loss': 0.355043888092041, 'eval_precision': 0.516504854368932, 'eval_recall': 0.5833333333333334, 'eval_f1': 0.5478887744593204, 'eval_accuracy': 0.8829787234042553, 'eval_runtime': 5.9397, 'eval_samples_per_second': 58.589, 'eval_steps_per_second': 1.852, 'epoch': 5.0}
Saved confusion matrix to: ./outputs_ner/confusion_matrix.npy


In [14]:
# 11 - PREDICTION .TEXT

def decode_predictions(tokens_batch, pred_ids_batch, labels_batch):
    lines = []
    for tokens, pred_ids_row, labels_row in zip(tokens_batch, pred_ids_batch, labels_batch):
        enc = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=cfg.max_length)
        word_ids = enc.word_ids()
        used_word_ids = set()
        word_level_preds, word_level_golds = [], []
        for p_i, l_i, w_i in zip(pred_ids_row, labels_row, word_ids):
            if w_i is None:
                continue
            if w_i not in used_word_ids:
                used_word_ids.add(w_i)
                gold = l_i if l_i != -100 else label2id.get("O", 0)
                word_level_preds.append(p_i)
                word_level_golds.append(gold)
        for tok, g, p in zip(tokens[:len(word_level_preds)], word_level_golds, word_level_preds):
            lines.append(f"{tok}\t{id2label[g]}\t{id2label[p]}")
        lines.append("[SENT_BREAK]")
    return "\n".join(lines)

predictions, labels, _ = trainer.predict(ds["test"])
pred_ids = np.argmax(predictions, axis=-1).tolist()
labels = labels.tolist()

pred_out_path = os.path.join(cfg.output_dir, "prediction.txt")
with open(pred_out_path, "w", encoding="utf-8") as f:
    f.write(decode_predictions(test_data["tokens"], pred_ids, labels))
print("Wrote:", pred_out_path)


/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


/opt/conda/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ATK seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Wrote: ./outputs_ner/prediction.txt


In [18]:
# Step 12  - Load bext checkpoint andd Quick Inference 

best_model_dir = trainer.state.best_model_checkpoint or cfg.output_dir
inference_model = AutoModelForTokenClassification.from_pretrained(best_model_dir)
inference_tokenizer = tokenizer

def tag(sentence: str):
    words = sentence.strip().split()
    enc = inference_tokenizer(words, is_split_into_words=True, return_tensors="pt")
    with torch.no_grad():
        logits = inference_model(**enc).logits
    pred = logits.argmax(-1)[0].tolist()
    word_ids = enc.word_ids()
    used = set()
    tags = []
    for p, w in zip(pred, word_ids):
        if w is None:
            continue
        if w not in used:
            used.add(w)
            tags.append(id2label[p])
    return list(zip(words, tags))

tag("The process is launched via cmd, with the code injecting data from the file into the running process")


[('The', 'O'),
 ('process', 'O'),
 ('is', 'O'),
 ('launched', 'O'),
 ('via', 'ATK'),
 ('cmd,', 'O'),
 ('with', 'O'),
 ('the', 'O'),
 ('code', 'ATK'),
 ('injecting', 'ATK'),
 ('data', 'ATK'),
 ('from', 'ATK'),
 ('the', 'ATK'),
 ('file', 'ATK'),
 ('into', 'ATK'),
 ('the', 'ATK'),
 ('running', 'ATK'),
 ('process', 'ATK')]